# LSTM Labeler Provenance and Checkpoint Verification

This notebook documents the provenance of the LSTM-derived silver labels. It reconstructs the stored architecture, checks the saved checkpoint against the labelled dataset, and evaluates that checkpoint only on the public SmSA source-domain test split.


## 1 Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level audit records, predictions, checkpoints, or intermediate artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the restricted provenance-check contract

Imports model, metric, and data utilities; defines restricted paths for the source notebook, checkpoint, vocabulary, and labelled dataset; and fixes the verification scope, batch size, random state, and device. The paths are intentionally placeholders.


In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.nn.utils.rnn import pad_sequence

try:
    from datasets import load_dataset
    import datasets
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Package datasets belum tersedia. Install di environment notebook yang dipakai untuk eksperimen.'
    ) from exc


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'lstm_provenance'
FIGURE_DIR = ROUND1_DIR / 'figures' / 'lstm_provenance'

SOURCE_NOTEBOOK = PROJECT_ROOT / 'notebooks' / '00_labeling_lstm.ipynb'
CHECKPOINT_PATH = PROJECT_ROOT / 'models' / 'lstm_labeler' / 'lstm_sentiment_model.pt'
VOCAB_PATH = PROJECT_ROOT / 'models' / 'lstm_labeler' / 'vocabulary.pkl'
LABELED_DATA_PATH = PROJECT_ROOT / 'data' / 'interim' / 'labeled_reviews_lstm.csv'

VERIFY_FULL_LABELLED_DATA = True
CONSISTENCY_SAMPLE_N = 256
BATCH_SIZE = 128
RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for required_path in [SOURCE_NOTEBOOK, CHECKPOINT_PATH, VOCAB_PATH, LABELED_DATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Artifact wajib tidak ditemukan: {required_path}')

print(f'Restricted project root placeholder: {PROJECT_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Device       : {DEVICE}')
print(f'Full check   : {VERIFY_FULL_LABELLED_DATA}')


### Reconstruct the LSTM architecture and text pipeline

Defines the vocabulary class, historical normalization and tokenization functions, and bidirectional LSTM architecture required to load the archived checkpoint without retraining it.


In [ ]:
import pickle
import re
from collections import Counter

import torch.nn as nn


class Vocabulary:

    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_freq = Counter()
        self.min_freq = min_freq

    def build_vocab(self, texts):
        for text in texts:
            self.word_freq.update(tokenize(preprocess_text(text)))
        idx = len(self.word2idx)
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq and word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, text):
        tokens = tokenize(preprocess_text(text))
        return [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]

    def __len__(self):
        return len(self.word2idx)


def preprocess_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text):
    return text.split()


class LSTMSentimentClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers,
                 bidirectional=True, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True,
        )
        self.fc_hidden = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc1 = nn.Linear(self.fc_hidden, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, text, attention_mask=None):
        embedded = self.dropout(self.embedding(text))
        lstm_out, _ = self.lstm(embedded)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)
            pooled = (lstm_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = lstm_out.mean(dim=1)
        return self.fc2(self.dropout(self.relu(self.fc1(self.dropout(pooled)))))


### Audit the source labelling notebook statically

Reads the original labelling notebook as JSON and records aggregate evidence about SmSA loading, test-split handling, checkpoint-selection code, sequence truncation, and labelled-output naming. This is provenance documentation, not a new training run.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


with SOURCE_NOTEBOOK.open(encoding='utf-8') as handle:
    source_nb = json.load(handle)

source_code = '\n'.join(
    ''.join(cell.get('source', []))
    for cell in source_nb.get('cells', [])
    if cell.get('cell_type') == 'code'
)

source_audit = {
    'source_notebook': str(SOURCE_NOTEBOOK.relative_to(PROJECT_ROOT)),
    'source_notebook_sha256': sha256_file(SOURCE_NOTEBOOK),
    'source_notebook_cells': len(source_nb.get('cells', [])),
    'smsa_loaded_in_source': "load_dataset(" in source_code and "'smsa'" in source_code,
    'smsa_test_split_loaded_in_source': "test_df = pd.DataFrame(smsa_dataset['test'])" in source_code,
    'explicit_smsa_test_loader_found': 'test_loader' in source_code or 'test_dataset' in source_code,
    'shallow_best_state_copy_pattern_found': 'best_model_state = model.state_dict().copy()' in source_code,
    'historical_inference_has_max_len_truncation': 'encoded = encoded[:MAX_LEN]' in source_code[source_code.find('def batch_predict'):],
    'historical_labeling_output': 'data/interim/labeled_reviews_lstm.csv',
}

with (OUTPUT_DIR / 'lstm_source_notebook_static_audit.json').open('w', encoding='utf-8') as handle:
    json.dump(source_audit, handle, indent=2)

print(json.dumps(source_audit, indent=2))
if source_audit['shallow_best_state_copy_pattern_found']:
    print('\nWARNING: shallow state_dict copy ditemukan. Verifikasi checkpoint aktual diperlukan sebelum menyebutnya best checkpoint.')
if not source_audit['explicit_smsa_test_loader_found']:
    print('WARNING: source notebook memuat SmSA test split tetapi tidak menemukan evaluasi test eksplisit.')


### Load and verify checkpoint metadata

Loads the restricted checkpoint and vocabulary, validates required metadata and label order, reconstructs the model, verifies vocabulary size, counts parameters, and writes aggregate checkpoint metadata.


In [ ]:
def load_checkpoint(path: Path, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)
    except Exception:
        return torch.load(path, map_location=device, weights_only=False)


checkpoint = load_checkpoint(CHECKPOINT_PATH, DEVICE)
with VOCAB_PATH.open('rb') as handle:
    vocab = pickle.load(handle)

required_checkpoint_keys = {
    'model_state_dict', 'vocab_size', 'embedding_dim', 'hidden_dim',
    'output_dim', 'n_layers', 'bidirectional', 'dropout', 'label_map'
}
missing_keys = sorted(required_checkpoint_keys.difference(checkpoint))
if missing_keys:
    raise KeyError(f'Checkpoint tidak memiliki key wajib: {missing_keys}')

id_to_label = {int(key): str(value).strip().lower() for key, value in checkpoint['label_map'].items()}
label_ids = sorted(id_to_label)
if label_ids != list(range(int(checkpoint['output_dim']))):
    raise ValueError(f'Label map tidak cocok dengan output_dim: {id_to_label}')

model = LSTMSentimentClassifier(
    vocab_size=int(checkpoint['vocab_size']),
    embedding_dim=int(checkpoint['embedding_dim']),
    hidden_dim=int(checkpoint['hidden_dim']),
    output_dim=int(checkpoint['output_dim']),
    n_layers=int(checkpoint['n_layers']),
    bidirectional=bool(checkpoint['bidirectional']),
    dropout=float(checkpoint['dropout']),
).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()

state_shapes = {name: list(value.shape) for name, value in checkpoint['model_state_dict'].items()}
checkpoint_summary = {
    'checkpoint_path': str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)),
    'checkpoint_sha256': sha256_file(CHECKPOINT_PATH),
    'vocabulary_path': str(VOCAB_PATH.relative_to(PROJECT_ROOT)),
    'vocabulary_sha256': sha256_file(VOCAB_PATH),
    'labeled_data_path': str(LABELED_DATA_PATH.relative_to(PROJECT_ROOT)),
    'labeled_data_sha256': sha256_file(LABELED_DATA_PATH),
    'checkpoint_config': {
        key: (checkpoint[key].item() if hasattr(checkpoint[key], 'item') else checkpoint[key])
        for key in ['vocab_size', 'embedding_dim', 'hidden_dim', 'output_dim', 'n_layers', 'bidirectional', 'dropout']
    },
    'label_map': {str(key): value for key, value in id_to_label.items()},
    'vocabulary_length': len(vocab),
    'vocabulary_length_matches_checkpoint': len(vocab) == int(checkpoint['vocab_size']),
    'parameter_count': int(sum(parameter.numel() for parameter in model.parameters())),
    'state_dict_tensor_shapes': state_shapes,
}

if not checkpoint_summary['vocabulary_length_matches_checkpoint']:
    raise ValueError('Ukuran vocabulary tidak cocok dengan checkpoint.')

with (OUTPUT_DIR / 'lstm_checkpoint_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(checkpoint_summary, handle, indent=2)

print(json.dumps({key: value for key, value in checkpoint_summary.items() if key != 'state_dict_tensor_shapes'}, indent=2))


### Define deterministic inference and serialization helpers

Defines batched probability inference, conversion from probability vectors to labels, and JSON-safe conversion used by the following verification steps.


In [ ]:
def predict_probabilities(texts, batch_size=BATCH_SIZE):
    all_probabilities = []
    model.eval()
    texts = [str(text) if not pd.isna(text) else '' for text in texts]

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            encoded_batch = []
            for text in batch_texts:
                encoded = vocab.encode(text)
                if not encoded:
                    encoded = [vocab.word2idx['<UNK>']]
                encoded_batch.append(torch.tensor(encoded, dtype=torch.long))

            padded = pad_sequence(encoded_batch, batch_first=True, padding_value=0).to(DEVICE)
            attention_mask = (padded != 0).float().to(DEVICE)
            logits = model(padded, attention_mask)
            all_probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())

    return np.vstack(all_probabilities)


def probabilities_to_labels(probabilities):
    prediction_ids = probabilities.argmax(axis=1)
    return prediction_ids, np.array([id_to_label[int(label_id)] for label_id in prediction_ids], dtype=object)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    return value


PROBA_COLUMN_BY_LABEL = {
    'positif': 'prob_positif',
    'netral': 'prob_netral',
    'negatif': 'prob_negatif',
}
print('Inference helper siap. Output class order:', [id_to_label[idx] for idx in label_ids])


### Compare checkpoint outputs with stored silver labels

Runs the archived checkpoint on either the authorized full labelled dataset or a fixed sample, compares predicted labels and probabilities with stored silver labels, and writes both a private row-level consistency file and an aggregate summary.


In [ ]:
labeled_df = pd.read_csv(LABELED_DATA_PATH)
required_columns = {'text_raw', 'sentiment_label', *PROBA_COLUMN_BY_LABEL.values()}
missing_columns = sorted(required_columns.difference(labeled_df.columns))
if missing_columns:
    raise KeyError(f'Kolom wajib tidak ditemukan di labeled CSV: {missing_columns}')

if VERIFY_FULL_LABELLED_DATA:
    consistency_df = labeled_df.copy()
    consistency_scope = 'full_labeled_dataset'
else:
    consistency_df = labeled_df.sample(
        n=min(CONSISTENCY_SAMPLE_N, len(labeled_df)),
        random_state=RANDOM_STATE,
    ).copy()
    consistency_scope = f'random_sample_{len(consistency_df)}'

consistency_df = consistency_df.reset_index().rename(columns={'index': 'source_row_id'})
checkpoint_probabilities = predict_probabilities(consistency_df['text_raw'].tolist())
prediction_ids, checkpoint_labels = probabilities_to_labels(checkpoint_probabilities)

ordered_probability_columns = [PROBA_COLUMN_BY_LABEL[id_to_label[idx]] for idx in label_ids]
saved_probabilities = consistency_df[ordered_probability_columns].to_numpy(dtype=float)
saved_labels = consistency_df['sentiment_label'].astype(str).str.strip().str.lower().to_numpy()

label_match = checkpoint_labels == saved_labels
absolute_probability_delta = np.abs(checkpoint_probabilities - saved_probabilities)

consistency_rows = pd.DataFrame({
    'source_row_id': consistency_df['source_row_id'],
    'saved_silver_label': saved_labels,
    'checkpoint_label': checkpoint_labels,
    'label_match': label_match,
    'max_abs_probability_delta': absolute_probability_delta.max(axis=1),
})
consistency_rows.to_csv(
    OUTPUT_DIR / 'lstm_checkpoint_csv_consistency_rows.csv',
    index=False,
    encoding='utf-8-sig',
)

consistency_summary = {
    'scope': consistency_scope,
    'n_rows': int(len(consistency_df)),
    'label_agreement': float(label_match.mean()),
    'n_label_mismatches': int((~label_match).sum()),
    'max_abs_probability_delta': float(absolute_probability_delta.max()),
    'mean_abs_probability_delta': float(absolute_probability_delta.mean()),
    'n_rows_delta_gt_1e_4': int((consistency_rows['max_abs_probability_delta'] > 1e-4).sum()),
    'n_rows_delta_gt_1e_3': int((consistency_rows['max_abs_probability_delta'] > 1e-3).sum()),
    'checkpoint_probability_order': [id_to_label[idx] for idx in label_ids],
    'saved_probability_columns_in_checkpoint_order': ordered_probability_columns,
}
with (OUTPUT_DIR / 'lstm_checkpoint_csv_consistency_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(json_ready(consistency_summary), handle, indent=2)

print(json.dumps(json_ready(consistency_summary), indent=2))
if consistency_summary['n_label_mismatches'] > 0:
    print('\nWARNING: checkpoint tidak menghasilkan seluruh silver label yang tersimpan. Jangan menyimpulkan checkpoint identik sebelum penyebab diperiksa.')
else:
    print('\nPASS: checkpoint menghasilkan label yang sama pada scope verifikasi.')


### Load and document the public SmSA test split

Retrieves the public SmSA test split, checks label compatibility with the checkpoint, and records source-domain dataset metadata including sequences longer than the model length.


In [ ]:
smsa_dataset = load_dataset('indonlp/indonlu', 'smsa', trust_remote_code=True)
smsa_test = pd.DataFrame(smsa_dataset['test'])

if not {'text', 'label'}.issubset(smsa_test.columns):
    raise KeyError(f'Kolom SmSA test tidak sesuai: {smsa_test.columns.tolist()}')

smsa_label_values = sorted(pd.Series(smsa_test['label']).dropna().unique().tolist())
expected_label_values = list(range(int(checkpoint['output_dim'])))
if smsa_label_values != expected_label_values:
    raise ValueError(
        f'Label SmSA test {smsa_label_values} tidak cocok dengan checkpoint output IDs {expected_label_values}.'
    )

label_feature = smsa_dataset['test'].features.get('label')
smsa_metadata = {
    'dataset': 'indonlp/indonlu',
    'configuration': 'smsa',
    'split': 'test',
    'n_test': int(len(smsa_test)),
    'label_values': smsa_label_values,
    'dataset_label_feature': str(label_feature),
    'checkpoint_label_map': {str(key): value for key, value in id_to_label.items()},
    'n_sequences_longer_than_128_tokens': int((smsa_test['text'].map(lambda text: len(vocab.encode(text))) > 128).sum()),
}
with (OUTPUT_DIR / 'smsa_test_dataset_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(json_ready(smsa_metadata), handle, indent=2)

print(json.dumps(json_ready(smsa_metadata), indent=2))


### Evaluate the checkpoint on SmSA only

Calculates accuracy, macro-F1, weighted F1, class report, confusion matrix, and a source-domain figure for SmSA. These values document checkpoint provenance and are not evidence of performance on Banjarbaru Google Maps reviews.


In [ ]:
smsa_probabilities = predict_probabilities(smsa_test['text'].tolist())
smsa_pred_ids, smsa_pred_labels = probabilities_to_labels(smsa_probabilities)
smsa_true_ids = smsa_test['label'].to_numpy(dtype=int)
class_names = [id_to_label[idx] for idx in label_ids]

smsa_accuracy = accuracy_score(smsa_true_ids, smsa_pred_ids)
smsa_macro_f1 = f1_score(smsa_true_ids, smsa_pred_ids, average='macro', zero_division=0)
smsa_weighted_f1 = f1_score(smsa_true_ids, smsa_pred_ids, average='weighted', zero_division=0)
smsa_cm = confusion_matrix(smsa_true_ids, smsa_pred_ids, labels=label_ids)
smsa_report = pd.DataFrame(
    classification_report(
        smsa_true_ids,
        smsa_pred_ids,
        labels=label_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
).T

smsa_metrics = {
    'evaluation_scope': 'SmSA held-out test split only',
    'n_test': int(len(smsa_test)),
    'accuracy': float(smsa_accuracy),
    'macro_f1': float(smsa_macro_f1),
    'weighted_f1': float(smsa_weighted_f1),
    'class_order': class_names,
    'true_label_distribution': {
        id_to_label[idx]: int((smsa_true_ids == idx).sum()) for idx in label_ids
    },
    'predicted_label_distribution': {
        id_to_label[idx]: int((smsa_pred_ids == idx).sum()) for idx in label_ids
    },
}

pd.DataFrame(
    smsa_cm,
    index=[f'true_{name}' for name in class_names],
    columns=[f'pred_{name}' for name in class_names],
).to_csv(OUTPUT_DIR / 'smsa_test_confusion_matrix.csv', encoding='utf-8-sig')
smsa_report.to_csv(OUTPUT_DIR / 'smsa_test_classification_report.csv', encoding='utf-8-sig')
with (OUTPUT_DIR / 'smsa_test_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(json_ready(smsa_metrics), handle, indent=2)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(smsa_cm, cmap='Blues')
fig.colorbar(image, ax=ax, label='Review count')
ax.set_xticks(range(len(class_names)), labels=[name.title() for name in class_names])
ax.set_yticks(range(len(class_names)), labels=[name.title() for name in class_names])
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('LSTM Performance on SmSA Test Split')
for row_idx in range(smsa_cm.shape[0]):
    for col_idx in range(smsa_cm.shape[1]):
        ax.text(col_idx, row_idx, str(smsa_cm[row_idx, col_idx]), ha='center', va='center')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'smsa_test_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(json.dumps(json_ready(smsa_metrics), indent=2))
print('\nInterpretasi: metrik ini hanya menilai source domain SmSA, bukan kualitas final pada Google Maps.')


### Write the provenance summary

Combines the source-code audit, checkpoint metadata, consistency result, SmSA evaluation, and runtime environment into an aggregate provenance summary. No LSTM retraining or relabelling occurs in this notebook.


In [ ]:
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scikit_learn': sklearn.__version__,
    'datasets': datasets.__version__,
    'device': str(DEVICE),
    'cuda_available': bool(torch.cuda.is_available()),
    'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

final_summary = {
    'purpose': 'Round 1 provenance and checkpoint verification only; no LSTM retraining or relabeling performed.',
    'source_static_audit': source_audit,
    'checkpoint_summary': checkpoint_summary,
    'checkpoint_csv_consistency': consistency_summary,
    'smsa_test_evaluation': smsa_metrics,
    'environment': environment,
    'interpretation_notes': [
        'SmSA test metrics are source-domain evidence and must not be reported as Google Maps performance.',
        'Google Maps downstream model metrics remain agreement with LSTM silver labels unless evaluated against final manual labels.',
        'A shallow best_model_state copy pattern in the source notebook requires careful checkpoint wording in the manuscript.',
    ],
}

with (OUTPUT_DIR / 'lstm_provenance_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(json_ready(final_summary), handle, indent=2)

print('R1_00 selesai. Output utama:')
for output_path in sorted(OUTPUT_DIR.glob('*')):
    print(' -', output_path.name)
print('Figure:')
for figure_path in sorted(FIGURE_DIR.glob('*')):
    print(' -', figure_path.name)
